In [23]:
import numpy as np
import matplotlib.pyplot as plt
import math

states = ['E', '5', 'I']
start = 'S'
end = 'X'

# Emission probabilities
E = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'G': 0.95},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

# Transition probabilities
T = {
    'S': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'X': 0.1}
}


In [24]:
def log_prob_path(path, seq):
    logp = math.log(T['S'][path[0]])
    for i in range(len(seq)):
        s = path[i]
        b = seq[i]
        logp += math.log(E[s][b])
        if i < len(seq) - 1:
            logp += math.log(T[s][path[i+1]])
        else:
            logp += math.log(T[s]['X'])
    return logp / math.log(10)  # log base 10

seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
print("Log-prob:",log_prob_path(path, seq))  # Expected: -41.22


Log-prob: -17.90147856487019


In [ ]:
def viterbi(obs):
    n = len(obs)
    dp = {s: [-1e9]*n for s in states}
    back = {s: ['']*n for s in states}

    dp['E'][0] = math.log(T['S']['E']) + math.log(E['E'][obs[0]])

    for i in range(1, n):
        for curr in states:
            if obs[i] not in E[curr]: continue
            for prev in states:
                if curr in T[prev]:
                    prob = dp[prev][i-1] + math.log(T[prev][curr]) + math.log(E[curr][obs[i]])
                    if prob > dp[curr][i]:
                        dp[curr][i] = prob
                        back[curr][i] = prev

    # End transition
    max_prob = -1e9
    last_state = ''
    for s in states:
        prob = dp[s][-1] + math.log(T[s].get('X', 1e-9))
        if prob > max_prob:
            max_prob = prob
            last_state = s

    # Backtrack
    path = [last_state]
    for i in range(n-1, 0, -1):
        path.insert(0, back[path[0]][i])

    return ''.join(path), max_prob / math.log(10)

# Run Viterbi
v_path, score = viterbi(seq)
print("Viterbi path:", v_path)
print("Log-Score:", round(score, 2))


In [ ]:
color_map = {'E': 'blue', '5': 'green', 'I': 'red'}
plt.figure(figsize=(12, 1))
for i, s in enumerate(v_path):
    plt.plot([i, i], [0, 1], color=color_map[s], linewidth=4)
plt.title("Viterbi V-Plot")
plt.yticks([])
plt.xticks(range(len(seq)), list(seq), fontsize=8)
plt.show()
